In [1]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError


class Point2D(BaseModel):
    x: float = 0
    y: float = 0

class Circle2D(BaseModel):
    center: Point2D
    radius: float = Field(default=1, gt=0)

In [2]:
c = Circle2D(center=Point2D(x=1, y=1), radius=2)

In [3]:
c

Circle2D(center=Point2D(x=1.0, y=1.0), radius=2.0)

In [4]:
c.model_dump()

{'center': {'x': 1.0, 'y': 1.0}, 'radius': 2.0}

In [5]:
c.model_dump_json()

'{"center":{"x":1.0,"y":1.0},"radius":2.0}'

In [6]:
data = {
    "center": {
        "x": 5, 
        "y": -5
    },
    "radius": 10
}

c = Circle2D.model_validate(data)
c

Circle2D(center=Point2D(x=5.0, y=-5.0), radius=10.0)

In [7]:
data = """
{
    "center": {
        "x": 5, 
        "y": -5
    },
    "radius": 10
}
"""

c = Circle2D.model_validate_json(data)
c

Circle2D(center=Point2D(x=5.0, y=-5.0), radius=10.0)

In [8]:
c.center

Point2D(x=5.0, y=-5.0)

In [9]:
json_data = """
{
    "firstName": "David",
    "lastName": "Hilbert",
    "contactInfo": {
        "email": "d.hilbert@spectral-theory.com",
        "homePhone": {
            "countryCode": 49,
            "areaCode": 551,
            "localPhoneNumber": 123456789
        }
    },
    "personalInfo": {
        "nationality": "German",
        "born": {
            "date": "1862-01-23",
            "place": {
                "city": "Konigsberg",
                "country": "Prussia"
            }
        },
        "died": {
            "date": "1943-02-14",
            "place": {
                "city": "Gottingen",
                "country": "Germany"
            }
        }
    },
    "awards": ["Lobachevsky Prize", "Bolyai Prize", "ForMemRS"],
    "notableStudents": ["von Neumann", "Weyl", "Courant", "Zermelo"]
}
"""

In [11]:
from pydantic import EmailStr, PastDate
from pydantic.alias_generators import to_camel
from typing import Optional


class ContactInfo(BaseModel):
    model_config = ConfigDict(extra="ignore")

    email: Optional[EmailStr] = None

class PlaceInfo(BaseModel):
    city: str
    country: str
    
class PlaceDateInfo(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    
    date_: PastDate = Field(alias="date")
    place: PlaceInfo
    
class PersonalInfo(BaseModel):
    model_config = ConfigDict(extra="ignore")

    nationality: str
    born: PlaceDateInfo

class Person(BaseModel):
    model_config = ConfigDict(alias_generator=to_camel, populate_by_name=True, extra="ignore")
    
    first_name: str
    last_name: str
    contact_info: ContactInfo
    personal_info: PersonalInfo
    notable_students: list[str] = []

In [12]:
p = Person.model_validate_json(json_data)
p

Person(first_name='David', last_name='Hilbert', contact_info=ContactInfo(email='d.hilbert@spectral-theory.com'), personal_info=PersonalInfo(nationality='German', born=PlaceDateInfo(date_=datetime.date(1862, 1, 23), place=PlaceInfo(city='Konigsberg', country='Prussia'))), notable_students=['von Neumann', 'Weyl', 'Courant', 'Zermelo'])

In [13]:
print(p.model_dump_json(by_alias=True, indent=2))

{
  "firstName": "David",
  "lastName": "Hilbert",
  "contactInfo": {
    "email": "d.hilbert@spectral-theory.com"
  },
  "personalInfo": {
    "nationality": "German",
    "born": {
      "date": "1862-01-23",
      "place": {
        "city": "Konigsberg",
        "country": "Prussia"
      }
    }
  },
  "notableStudents": [
    "von Neumann",
    "Weyl",
    "Courant",
    "Zermelo"
  ]
}
